# 00 — Workspace Setup

Idempotent. Run this first, and re-run it any time you come back to a fresh workspace.

Creates the Unity Catalog namespace, verifies the serverless environment supports Spark ML,
and checks that the source data has been uploaded.

**Manual prerequisites (not automatable, see README):**

1. Databricks Free Edition account with LinkedIn verification completed
2. Serverless **environment version 4** selected in the Environment side panel
3. `flights_sample_3m.csv` uploaded to `/Volumes/workspace/flights/raw/`

This notebook contains no credentials and no personal paths. Anything that would print
credential material belongs in a scratch notebook outside the Git folder.


In [ ]:
# Environment probe — env v4 is required for pyspark.ml and mlflow.spark on serverless
import sys

import mlflow
import pyspark

print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"MLflow  : {mlflow.__version__}")

try:
    import pyspark.ml  # noqa: F401
    import mlflow.spark  # noqa: F401

    print("\nSpark ML available — environment version 4 confirmed")
except ImportError as e:
    raise RuntimeError(
        "Spark ML unavailable. Open the Environment side panel and set environment version 4."
    ) from e


In [ ]:
# Namespace DDL — safe to re-run
CATALOG = "workspace"
SCHEMA = "flights"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.artifacts")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Namespace ready: {CATALOG}.{SCHEMA}")
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA}"))


In [ ]:
# Preflight — confirm the source CSV landed in the volume
RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
SOURCE_FILE = "flights_sample_3m.csv"

files = {f.name: f.size for f in dbutils.fs.ls(RAW_VOLUME)}
if SOURCE_FILE not in files:
    raise FileNotFoundError(
        f"{SOURCE_FILE} not found in {RAW_VOLUME}.\n"
        f"Found: {sorted(files) or '(empty)'}\n\n"
        "Upload it via Catalog Explorer > workspace > flights > raw > Upload to this volume."
    )

size_mb = files[SOURCE_FILE] / 1024**2
print(f"{SOURCE_FILE}  ({size_mb:,.1f} MB)")

# Peek at the header without loading the file
header = spark.read.csv(f"{RAW_VOLUME}/{SOURCE_FILE}", header=True).limit(5)
print(f"Columns: {len(header.columns)}")
display(header)


In [ ]:
# Secret preflight — the live-scoring path (06_api_ingest) needs this scope.
# Non-fatal: the Bronze->Gold->train path does not touch the API at all, so a
# missing secret should not block setup.
#
# Deliberately prints only whether the key resolves, never the key or its
# length — a length is a small but free leak in a public repo.
from src import config

try:
    _key = dbutils.secrets.get(config.AVIATIONSTACK_SECRET_SCOPE, config.AVIATIONSTACK_SECRET_KEY)
    status = "configured" if _key else "present but EMPTY"
    del _key
except Exception as e:
    status = f"NOT configured ({type(e).__name__})"

print(f"Secret {config.AVIATIONSTACK_SECRET_SCOPE}/{config.AVIATIONSTACK_SECRET_KEY}: {status}")
if not status.startswith("configured"):
    print(
        "\n  Only 06_api_ingest and 07_score need this. To set it up, run from the CLI:\n"
        f"    databricks secrets create-scope {config.AVIATIONSTACK_SECRET_SCOPE}\n"
        f"    databricks secrets put-secret {config.AVIATIONSTACK_SECRET_SCOPE} "
        f"{config.AVIATIONSTACK_SECRET_KEY}"
    )


In [ ]:
# Summary — paste this output into the RUNBOOK reply block
print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"  Catalog / schema : {CATALOG}.{SCHEMA}")
print("  Volumes          : raw, artifacts")
print(f"  Source file      : {SOURCE_FILE} ({size_mb:,.1f} MB)")
print(f"  Spark            : {pyspark.__version__}")
print(f"  MLflow           : {mlflow.__version__}")
print("\nNext: notebooks/01_bronze.ipynb")
